In [ ]:
# Imports
import pandas as pd
import os, re, unicodedata, numpy as np, pandas as pd
from pathlib import Path
from calendar import month_abbr
from google.colab import drive

# Mount Drive
if os.path.exists("/content/drive"):
    try:
        !fusermount -u /content/drive 2>/dev/null
        !rm -rf /content/drive
    except Exception as e:
        print("Unmount warning:", e)
drive.mount('/content/drive')

# Paths
NBA_PATH = '/content/drive/MyDrive/nba-draft-sucess/final_master/nba_stats_with_labels.csv'
COLLEGE_COMBINE_PATH = '/content/drive/MyDrive/nba-draft-sucess/final_master/college_combine_features_matched.csv'
OUT_PATH = '/content/drive/MyDrive/nba-draft-sucess/final_master/final_training_dataset.csv'

In [ ]:
# Load
nba = pd.read_csv(NBA_PATH, dtype=str)
college = pd.read_csv(COLLEGE_COMBINE_PATH, dtype=str)

# Make sure join_key is present and normalized
nba["join_key"] = nba["Player"].str.strip().str.lower()
college["join_key"] = college["join_key"].str.strip().str.lower()

# Keep only intersection of players
common_keys = set(nba["join_key"]) & set(college["join_key"])
nba_final = nba[nba["join_key"].isin(common_keys)].copy()
college_final = college[college["join_key"].isin(common_keys)].copy()

print(f" Players kept: {len(common_keys)} / {len(nba)} total")
print(f"Dropped {len(nba) - len(common_keys)} NBA players without college+combine data")

# Merge to get full aligned training data (features + NBA targets)
final_df = pd.merge(college_final, nba_final, on="join_key", how="inner", suffixes=("_college", "_nba"))
final_df.to_csv(OUT_PATH, index=False)

print(f" Saved aligned training dataset to {OUT_PATH}")
print(f"Rows: {len(final_df)}, Columns: {len(final_df.columns)}")
